In [0]:
import sys, os

# Add the repo root to sys.path so we can import from src/
# This works because Databricks Git folders live under /Workspace/Repos/...
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
repo_root = "/Workspace" + "/".join(notebook_path.split("/")[:-2])

if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

print(f"Repo root added to sys.path: {repo_root}")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    DoubleType, IntegerType
)

from src.utils.spark_session import get_spark_session
from src.quality.dq_checks import flag_quarantined_records

spark = get_spark_session(app_name = "BronzeIngestion")

In [0]:
# ── Cell 3: Load config ──────────────────────────────────
import yaml
import os

# Widget lets you switch source file without changing code
# In ADF production runs, this gets passed as a pipeline parameter
dbutils.widgets.text("source_file", cfg["source_file"] if "cfg" in dir() else "creditcard.csv")
source_file_override = dbutils.widgets.get("source_file")

config_path = os.path.join(repo_root, "config", "pipeline_config.yml")

with open(config_path, "r") as f:
    config = yaml.safe_load(f)

env    = "dev"
cfg    = config["environments"][env]
dq_cfg = config["data_quality"]

# Use widget value if provided, otherwise fall back to config
source_file = source_file_override if source_file_override else cfg["source_file"]

SOURCE_PATH = f"abfss://{cfg['container_raw']}@{cfg['storage_account']}.dfs.core.windows.net/{source_file}"
BRONZE_PATH = cfg["bronze_path"]

print(f"Config loaded successfully")
print(f"Environment : {env}")
print(f"Source file : {source_file}")
print(f"Source path : {SOURCE_PATH}")
print(f"Bronze path : {BRONZE_PATH}")

In [0]:
# ── Cell 4: ADLS Authentication via Service Principal ───────────────

# Read secrets from Key Vault via Databricks secret scope
# These never appear in plain text — dbutils.secrets.get() 
# returns a redacted value if you try to print it

from src.utils.spark_session import configure_adls_oauth

client_id     = dbutils.secrets.get(scope="kv-bank-etl-scope", 
                                     key="databricks-sp-client-id")
client_secret = dbutils.secrets.get(scope="kv-bank-etl-scope", 
                                     key="databricks-sp-client-secret")
tenant_id     = dbutils.secrets.get(scope="kv-bank-etl-scope", 
                                     key="databricks-sp-tenant-id")

storage_account = cfg["storage_account"]

configure_adls_oauth(spark, storage_account, client_id, client_secret, tenant_id)

print(f"Storage account : {storage_account}")

In [0]:
bronze_schema = StructType([
    StructField("Time",   DoubleType(),  True),
    StructField("V1",     DoubleType(),  True),
    StructField("V2",     DoubleType(),  True),
    StructField("V3",     DoubleType(),  True),
    StructField("V4",     DoubleType(),  True),
    StructField("V5",     DoubleType(),  True),
    StructField("V6",     DoubleType(),  True),
    StructField("V7",     DoubleType(),  True),
    StructField("V8",     DoubleType(),  True),
    StructField("V9",     DoubleType(),  True),
    StructField("V10",    DoubleType(),  True),
    StructField("V11",    DoubleType(),  True),
    StructField("V12",    DoubleType(),  True),
    StructField("V13",    DoubleType(),  True),
    StructField("V14",    DoubleType(),  True),
    StructField("V15",    DoubleType(),  True),
    StructField("V16",    DoubleType(),  True),
    StructField("V17",    DoubleType(),  True),
    StructField("V18",    DoubleType(),  True),
    StructField("V19",    DoubleType(),  True),
    StructField("V20",    DoubleType(),  True),
    StructField("V21",    DoubleType(),  True),
    StructField("V22",    DoubleType(),  True),
    StructField("V23",    DoubleType(),  True),
    StructField("V24",    DoubleType(),  True),
    StructField("V25",    DoubleType(),  True),
    StructField("V26",    DoubleType(),  True),
    StructField("V27",    DoubleType(),  True),
    StructField("V28",    DoubleType(),  True),
    StructField("Amount", DoubleType(),  True),
    StructField("Class",  IntegerType(), True),
])

print(f"Schema defined with {len(bronze_schema.fields)} fields")

In [0]:
df_raw = (
    spark.read
    .format("csv")
    .schema(bronze_schema)
    .option("header", "true")
    .option("mode", "PERMISSIVE")
    .load(SOURCE_PATH)
)

row_count = df_raw.count()
print(f"Raw row count: {row_count:,}")
df_raw.printSchema()

In [0]:
# ── Cell 7: Add audit metadata columns ──────────────────

pipeline_run_id = "manual_run_001"

df_with_meta = (
    df_raw \
        .withColumn("_ingestion_timestamp", F.current_timestamp()) \
        .withColumn("_ingestion_date", F.current_date()) \
        .withColumn("_pipeline_run_id", F.lit(pipeline_run_id)) \
        .withColumn("_source_file", F.lit(cfg["source_file"])) \
        .withColumn("_environment", F.lit(env))
)

print("Metadata columns added: ")
for col in df_with_meta.columns:
  if col.startswith("_"):
    print(col)

In [0]:
# ── Cell 8: Flag bad records using DQ module ────────────

df_flagged = flag_quarantined_records(
    df_with_meta,
    min_amount= dq_cfg["min_amount"],
    max_amount= dq_cfg["max_amount"],
    max_time= dq_cfg["max_time_value"]
)

total_count = df_flagged.count()
quarantine_count = df_flagged.filter(F.col("_is_quarantined") == True).count()
valid_record_count = total_count - quarantine_count

print("=" * 50)
print(f"  Total records count : {total_count:,}")
print(f"  Clean records    : {valid_record_count:,}")
print(f"  Quarantined      : {quarantine_count:,}")
print(f"  Quarantine rate  : {quarantine_count/total_count*100:.2f}%")
print("=" * 50)

df_flagged.filter(F.col("_is_quarantined") == True).show(5, truncate=False)

In [0]:
# ── Cell 9: Write to Bronze Delta table ─────────────────

df_flagged.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "false") \
    .partitionBy("_ingestion_date") \
    .save(BRONZE_PATH)

print(f"Successfully written to Bronze")
print(f"Location: {BRONZE_PATH}")

# Immediately verify the write
df_verify = spark.read.format("delta").load(BRONZE_PATH)
print(f"Bronze table total rows: {df_verify.count():,}")

In [0]:
# ── Cell 10: Delta table history ────────────────────────
# This is your audit trail — every write operation recorded

display(
    spark.sql(f"DESCRIBE HISTORY delta.`{BRONZE_PATH}`")
)

print("\nPartition distribution:")
df_verify.groupBy("_ingestion_date").count().show()

print("Quarantine distribution:")
df_verify.groupBy("_is_quarantined").count().show()